# Corrected copy

All fixes are marked `# FIX:` in the code. Changes were checked on the Python 3.7 grammar (no walrus, no f-string `=`, no type-hint generics); the **sanity-test cell at the bottom** exercises every function.

**Behaviour changes you should know about (conventions made consistent):**
- "Not found" is now `-1` for searches (array indices) and `None` for trees / hash tables / empty stack or queue. `False` was ambiguous because `False == 0`.
- `hex_to_dex` renamed `hex_to_dec`; `quick_sort_haore` renamed `quick_sort_hoare`.
- Both in-place quick sorts now use a random pivot (needs `import random`, done in the Quick Sort cell, so run cells top to bottom).
- New `TreeNode` class cell added before the BSTs.
- `CQueue` now keeps `head` = front index, `tail` = next free slot, and updates `count`.
- Linked lists / hash tables: `insert_sorted` accepts duplicates; hash tables update an existing key rather than duplicating it.
- `delete()` methods return `True`/`False`.

# Data Representation

## Binary Numbers

In [ ]:
# Denary/Decimal to Binary
def dec_to_bin(dec):
    """Return the binary string for a non-negative integer."""
    if dec < 0:                                 # FIX: negative input looped forever
        raise ValueError("dec must be non-negative")

    bits = ""                                   # FIX: was `bin` (shadowed built-in)
    while dec:
        bits = str(dec % 2) + bits
        dec //= 2

    return bits or "0"

In [ ]:
# Binary to Denary/Decimal
def bin_to_dec(bits):
    """Return the integer value of a binary string such as '1011'."""
    dec = 0

    for bit in bits:                            # FIX: was `chr` (shadowed built-in)
        if bit not in "01":
            raise ValueError("not a binary digit: %r" % bit)
        dec = dec * 2 + int(bit)                # FIX: int(bit) - was adding a str

    return dec

## Hexadecimals

In [ ]:
# Denary/Decimal to Hexadecimal
def dec_to_hex(dec):
    """Return the uppercase hexadecimal string for a non-negative integer."""
    if dec < 0:
        raise ValueError("dec must be non-negative")

    digits = ""                                 # FIX: was `hex` (shadowed built-in)
    hex_digits = "0123456789ABCDEF"

    while dec:
        digits = hex_digits[dec % 16] + digits
        dec //= 16

    return digits or "0"                        # FIX: 0 now gives "0", not ""

In [ ]:
# Hexadecimal to Denary/Decimal
def hex_to_dec(hex_str):                        # FIX: name typo (hex_to_dex)
    """Return the integer value of a hex string such as '1F' (case-insensitive)."""
    dec = 0
    hex_digits = "0123456789ABCDEF"

    for digit in hex_str.upper():               # FIX: .upper() accepts lowercase
        dec = dec * 16 + hex_digits.index(digit)  # FIX: .index() raises on bad digit; .find() returned -1

    return dec

# Fundamental Algorithms

In [ ]:
unsorted_test_lst = [12,1231,1,23,36,453,2234,3125,43654,4,786,243,13,4,65,3376,35687,33,2]
test_lst = sorted(unsorted_test_lst)            # FIX: derived, so it can never drift from the input

## Linear Search

In [ ]:
# Linear Search
def linear_search(array, seek):
    """Return the index of seek, or -1 if absent."""
    for i in range(len(array)):
        if array[i] == seek:
            return i

    return -1                                   # FIX: was False, which equals 0 (a valid index)

print(linear_search(test_lst, 6767))   # -1
print(linear_search(test_lst, 1))      # 0
print(linear_search(test_lst, 43654))  # 18

## Binary Search

In [ ]:
# Binary Search - Iterative (array must be sorted)
def binary_search_iter(array, seek):
    """Return the index of seek, or -1 if absent."""
    lo = 0
    hi = len(array) - 1

    while lo <= hi:
        mid = (lo + hi) // 2
        if array[mid] == seek:
            return mid
        elif seek > array[mid]:
            lo = mid + 1
        else:
            hi = mid - 1

    return -1                                   # FIX: was False

print(binary_search_iter(test_lst, 6767))   # -1
print(binary_search_iter(test_lst, 1))      # 0
print(binary_search_iter(test_lst, 43654))  # 18

In [ ]:
# Binary Search - Recursive (array must be sorted)
def binary_search_rec(array, seek, lo=None, hi=None):
    """Return the index of seek, or -1 if absent."""
    if lo is None and hi is None:
        lo = 0
        hi = len(array) - 1

    if lo > hi:
        return -1                               # FIX: was False
    mid = (lo + hi) // 2

    if seek == array[mid]:
        return mid
    elif seek > array[mid]:
        return binary_search_rec(array, seek, mid + 1, hi)
    else:
        return binary_search_rec(array, seek, lo, mid - 1)

print(binary_search_rec(test_lst, 6767))   # -1
print(binary_search_rec(test_lst, 1))      # 0
print(binary_search_rec(test_lst, 43654))  # 18

## Bubble Sort

In [ ]:
# Bubble Sort
def bubble_sort(array):
    for i in range(len(array)-1):
        swap = False
        for j in range(len(array)-i-1):
            if array[j+1] < array[j]:
                array[j+1],array[j] = array[j],array[j+1]
                swap = True
        if not swap:
            break

bubble_lst = unsorted_test_lst[:]
bubble_sort(bubble_lst)
print(bubble_lst)
print(bubble_lst == test_lst)

## Insertion Sort

In [ ]:
# Insertion Sort
def insertion_sort(array):
    for i in range(1, len(array)):              # FIX: start at 1 (index 0 is trivially sorted)
        j = i
        while j > 0 and array[j-1] > array[j]:
            array[j-1], array[j] = array[j], array[j-1]
            j -= 1

insertion_lst = unsorted_test_lst[:]
insertion_sort(insertion_lst)
print(insertion_lst)
print(insertion_lst == test_lst)

## Merge Sort

In [ ]:
# Merge Sort
def merge_sort(array):
    if len(array) <= 1:
        return array

    mid = len(array) // 2

    left = merge_sort(array[:mid])
    right = merge_sort(array[mid:])

    return merge(left, right)

def merge(left, right):
    merged = []
    i = j = 0

    while i < len(left) and j < len(right):
        if left[i] <= right[j]:                 # FIX: <= keeps equal items in original order (stable)
            merged.append(left[i])
            i += 1
        else:
            merged.append(right[j])
            j += 1

    merged += left[i:] + right[j:]

    return merged

merge_lst = merge_sort(unsorted_test_lst)
print(merge_lst)
print(merge_lst == test_lst)

## Quick Sort

In [ ]:
# Quick Sort (not in place)
import random                                   # FIX: moved out of the recursive function

def quick_sort(array):
    if len(array) <= 1:                         # FIX: was <= 0
        return array

    pivot = random.choice(array)

    lesser = [i for i in array if i < pivot]
    equal = [i for i in array if i == pivot]
    greater = [i for i in array if i > pivot]

    return quick_sort(lesser) + equal + quick_sort(greater)

quick_lst = quick_sort(unsorted_test_lst)
print(quick_lst)
print(quick_lst == test_lst)

## Quick Sort - In Place

In [ ]:
# Quick Sort In Place - Lomuto's Partition
def quick_sort_lomuto(array):
    _quick_sort_lomuto(array, 0, len(array) - 1)

def _quick_sort_lomuto(array, lo, hi):
    if lo < hi:
        p = lomuto_partition(array, lo, hi)     # FIX: renamed from `mid` (it is the pivot's final index)
        _quick_sort_lomuto(array, lo, p - 1)
        _quick_sort_lomuto(array, p + 1, hi)

def lomuto_partition(array, lo, hi):
    # Random pivot avoids the O(n^2) / deep-recursion case on already-sorted input.
    r = random.randint(lo, hi)                  # FIX: was always array[hi]
    array[r], array[hi] = array[hi], array[r]
    pivot = array[hi]

    i = lo - 1
    for j in range(lo, hi):
        if array[j] < pivot:
            i += 1
            array[i], array[j] = array[j], array[i]
    array[hi], array[i+1] = array[i+1], array[hi]

    return i + 1

lomuto_lst = unsorted_test_lst[:]
quick_sort_lomuto(lomuto_lst)
print(lomuto_lst)
print(lomuto_lst == test_lst)

In [ ]:
# Quick Sort In Place - Hoare's Partition
def quick_sort_hoare(array):                    # FIX: typo haore -> hoare
    _quick_sort_hoare(array, 0, len(array) - 1)

def _quick_sort_hoare(array, lo, hi):
    if lo < hi:
        split = hoare_partition(array, lo, hi)  # FIX: renamed from `mid`
        _quick_sort_hoare(array, lo, split)     # Hoare: left part INCLUDES the split index
        _quick_sort_hoare(array, split + 1, hi)

def hoare_partition(array, lo, hi):
    r = random.randint(lo, hi)                  # FIX: random pivot (was array[lo])
    array[lo], array[r] = array[r], array[lo]
    pivot = array[lo]

    i = lo - 1
    j = hi + 1

    while True:
        # emulate do-while: always advance once, then keep going while on the wrong side
        i += 1
        while array[i] < pivot:
            i += 1

        j -= 1
        while array[j] > pivot:
            j -= 1

        if i >= j:
            return j
        array[i], array[j] = array[j], array[i]

hoare_lst = unsorted_test_lst[:]
quick_sort_hoare(hoare_lst)
print(hoare_lst)
print(hoare_lst == test_lst)

# Abstract Data Types

## Stack

In [ ]:
# Stack
class Stack:
    def __init__(self, size):
        self.size = size
        self.array = [None] * size
        self.tail = 0                           # index of the next free slot

    def is_empty(self):
        return self.tail == 0

    def is_full(self):
        return self.tail == self.size

    def push(self, val):
        """Return True if pushed, False if the stack is full."""
        if self.is_full():
            return False
        self.array[self.tail] = val
        self.tail += 1
        return True                             # FIX: was returning None on success

    def pop(self):                              # FIX: removed stray `val` parameter
        """Return the top item, or None if the stack is empty."""
        if self.is_empty():
            return None                         # FIX: None can't be confused with a stored 0/False
        self.tail -= 1
        return self.array[self.tail]

    def peek(self):
        """Return the top item without removing it, or None if empty."""
        if self.is_empty():                     # FIX: array[-1] used to be returned here
            return None
        return self.array[self.tail - 1]

## Queue

In [ ]:
# Circular Queue
class CQueue:
    def __init__(self, size):
        self.size = size
        self.count = 0
        self.q = [None] * size
        self.head = 0                           # index of the FRONT item
        self.tail = 0                           # FIX: index of the next free slot (was -1)

    def is_empty(self):
        return self.count == 0

    def is_full(self):
        return self.count == self.size

    def enqueue(self, val):
        """Return True if added, False if the queue is full."""
        if self.is_full():
            return False

        self.q[self.tail] = val
        self.tail = (self.tail + 1) % self.size
        self.count += 1                         # FIX: count was never updated
        return True

    def dequeue(self):
        """Return the front item, or None if the queue is empty."""
        if self.is_empty():
            return None

        val = self.q[self.head]                 # FIX: read BEFORE moving head
        self.q[self.head] = None
        self.head = (self.head + 1) % self.size
        self.count -= 1                         # FIX: count was never updated
        return val

    def display(self):
        """Return the items from front to back (empty list if empty)."""
        return [self.q[(self.head + i) % self.size] for i in range(self.count)]

## Linked List

In [ ]:
# Linked List (singly linked; insert_sorted keeps order if you only use it)
class LinkedListNode:
    def __init__(self, data):
        self.data = data
        self.next = None

class LinkedList:
    def __init__(self):
        self.root = None

    def insert_first(self, node):
        node.next = self.root
        self.root = node

    def insert_last(self, node):
        if self.root is None:
            self.root = node
        else:
            curr = self.root
            while curr.next is not None:
                curr = curr.next
            curr.next = node

    def insert_sorted(self, node):
        if self.root is None or node.data <= self.root.data:   # FIX: node.data (was node), and <=
            node.next = self.root
            self.root = node
        else:
            prev = self.root                    # FIX: prev can no longer be None here
            curr = self.root.next
            while curr is not None and curr.data < node.data:
                prev = curr
                curr = curr.next

            prev.next = node
            node.next = curr

    def delete(self, data):
        """Delete the first node holding data. Return True if found."""
        prev = None
        curr = self.root
        while curr is not None:                 # FIX: was curr.next - never checked the last node
            if curr.data == data:
                if prev is None:
                    self.root = curr.next
                else:
                    prev.next = curr.next
                return True
            prev = curr
            curr = curr.next

        return False                            # FIX: one consistent return convention

In [ ]:
# Array Based Sorted Linked List
class ArrayBasedLinkedListNode:
    def __init__(self, data):
        self.data = data
        self.next = -1

class ArrayBasedLinkedList:
    def __init__(self, size):
        # FIX: list comprehension -> `size` DISTINCT nodes (was [node]*size = one shared node)
        self.array = [ArrayBasedLinkedListNode(None) for _ in range(size)]
        for i in range(size - 1):
            self.array[i].next = i + 1          # chain every slot onto the free list
        self.root = -1                          # -1 means "empty list"
        self.free = 0 if size > 0 else -1       # head of the free list

    def is_full(self):
        return self.free == -1

    def insert_sorted(self, data):
        """Return True if inserted, False if there is no free slot."""
        if self.is_full():                      # FIX: guard added
            return False

        idx = self.free
        node = self.array[idx]
        node.data = data
        self.free = node.next

        if self.root == -1 or self.array[self.root].data >= data:   # FIX: >= (was >)
            node.next = self.root
            self.root = idx
        else:
            prev = self.root
            curr = self.array[prev].next
            while curr != -1 and self.array[curr].data < data:
                prev = curr
                curr = self.array[curr].next

            self.array[prev].next = idx
            node.next = curr

        return True

    def delete(self, data):
        """Delete the first node holding data. Return True if found."""
        prev = -1
        curr = self.root

        while curr != -1:
            if self.array[curr].data == data:   # FIX: was != (inverted)
                if prev == -1:                  # deleting the root
                    self.root = self.array[curr].next   # FIX: old index (curr) is kept for recycling
                else:
                    self.array[prev].next = self.array[curr].next   # FIX: assign .next, not the node

                self.array[curr].data = None    # return the slot to the free list
                self.array[curr].next = self.free
                self.free = curr
                return True

            prev = curr
            curr = self.array[curr].next

        return False

## Binary Search Tree

In [ ]:
# FIX (new cell): node class used by both BST implementations - it was missing from the notebook
class TreeNode:
    def __init__(self, data):
        self.data = data
        self.left = None
        self.right = None

In [ ]:
# Binary Search Tree - Iterative
class BinarySearchTreeIter:
    def __init__(self):
        self.root = None

    def insert(self, node):
        if self.root is None:
            self.root = node
        else:
            curr = self.root
            while True:
                if node.data > curr.data:
                    if curr.right is None:
                        curr.right = node
                        return
                    curr = curr.right
                else:
                    if curr.left is None:
                        curr.left = node
                        return
                    curr = curr.left

    def search(self, data):
        curr = self.root
        while curr is not None:
            if curr.data == data:
                return curr
            elif data > curr.data:
                curr = curr.right
            else:
                curr = curr.left

        return None                             # FIX: None for "not found" (consistent with the hash tables)

    def inorder(self):
        result = []                             # FIX: [] for an empty tree (was False)
        stack = []
        curr = self.root

        while curr is not None or stack:
            while curr is not None:
                stack.append(curr)
                curr = curr.left

            curr = stack.pop()
            result.append(curr.data)
            curr = curr.right

        return result

    def preorder(self):
        if self.root is None:
            return []
        result = []
        stack = [self.root]

        while stack:
            curr = stack.pop()
            result.append(curr.data)

            if curr.right is not None:          # push right first so left is processed first
                stack.append(curr.right)
            if curr.left is not None:
                stack.append(curr.left)

        return result

    def postorder(self):
        result = []
        stack = []
        curr = self.root
        last = None                             # most recently visited node

        while curr is not None or stack:
            while curr is not None:
                stack.append(curr)              # FIX: push curr (was curr.left)
                curr = curr.left

            peek = stack[-1]

            if peek.right is not None and last is not peek.right:
                curr = peek.right               # right subtree not done yet
            else:
                result.append(peek.data)
                last = stack.pop()              # FIX: pop() with brackets

        return result

In [ ]:
# Binary Search Tree - Recursive
class BinarySearchTreeRec:
    def __init__(self):
        self.root = None

    def insert(self, node):
        self.root = self._insert(node, self.root)

    def _insert(self, node, root):
        if root is None:
            return node

        if node.data > root.data:
            root.right = self._insert(node, root.right)
        else:
            root.left = self._insert(node, root.left)

        return root

    def search(self, data):
        return self._search(data, self.root)

    def _search(self, data, root):
        if root is None:
            return None                         # FIX: None for "not found"

        if root.data == data:
            return root
        elif data > root.data:
            return self._search(data, root.right)
        else:
            return self._search(data, root.left)

    def inorder(self):
        result = []
        self._inorder(self.root, result)
        return result

    def _inorder(self, root, result):
        if root:
            self._inorder(root.left, result)    # FIX: pass result
            result.append(root.data)
            self._inorder(root.right, result)   # FIX: pass result

    def preorder(self):
        result = []
        self._preorder(self.root, result)
        return result

    def _preorder(self, root, result):
        if root:
            result.append(root.data)
            self._preorder(root.left, result)   # FIX: pass result
            self._preorder(root.right, result)  # FIX: pass result

    def postorder(self):
        result = []
        self._postorder(self.root, result)
        return result

    def _postorder(self, root, result):
        if root:
            self._postorder(root.left, result)
            self._postorder(root.right, result)
            result.append(root.data)

## Hash Table

In [ ]:
# Hash Table (Linear Probing)
class HashTableLP:
    def __init__(self, size):
        self.size = size
        self.buckets = [None] * size

    def _hash(self, key):
        # NOTE: str hashes are randomised per run (PYTHONHASHSEED); fine in memory only.
        return hash(key) % self.size

    def insert(self, key, value):
        """Insert or update. Return False only if the table is full."""
        idx = ref = self._hash(key)

        while self.buckets[idx] is not None:
            if self.buckets[idx][0] == key:     # FIX: update existing key instead of duplicating
                self.buckets[idx] = (key, value)
                return True
            idx = (idx + 1) % self.size
            if idx == ref:
                return False

        self.buckets[idx] = (key, value)
        return True

    def search(self, key):
        """Return the value for key, or None if absent."""
        idx = ref = self._hash(key)

        while self.buckets[idx] is not None:
            if self.buckets[idx][0] == key:
                return self.buckets[idx][1]
            idx = (idx + 1) % self.size
            if idx == ref:
                break

        return None                             # FIX: was False (== 0)

In [ ]:
# Hash Table (Separate Chaining)
class HashTableSC:
    def __init__(self, size):
        self.size = size
        self.buckets = [[] for _ in range(size)]

    def _hash(self, key):
        return hash(key) % self.size

    def insert(self, key, value):
        """Insert or update. Always returns True (chains never fill up)."""
        chain = self.buckets[self._hash(key)]

        for n, (k, _) in enumerate(chain):      # FIX: update existing key instead of duplicating
            if k == key:
                chain[n] = (key, value)
                return True
        chain.append((key, value))
        return True                             # FIX: same return convention as HashTableLP

    def search(self, key):
        """Return the value for key, or None if absent."""
        for k, v in self.buckets[self._hash(key)]:   # FIX: simple for-loop
            if k == key:
                return v

        return None                             # FIX: was False (== 0)

## Sanity tests

In [ ]:
# Sanity tests for every corrected function (all should pass silently on Python 3.7+)
assert dec_to_bin(0) == "0" and dec_to_bin(10) == "1010"
assert bin_to_dec("1010") == 10 and bin_to_dec("0") == 0
assert dec_to_hex(0) == "0" and dec_to_hex(255) == "FF"
assert hex_to_dec("ff") == 255 and hex_to_dec("1A") == 26
for n in range(0, 300):
    assert bin_to_dec(dec_to_bin(n)) == n and hex_to_dec(dec_to_hex(n)) == n

for f in (linear_search, binary_search_iter, binary_search_rec):
    assert f(test_lst, 1) == 0 and f(test_lst, 43654) == 18 and f(test_lst, 6767) == -1

assert bubble_lst == insertion_lst == merge_lst == quick_lst == lomuto_lst == hoare_lst == test_lst
big = list(range(3000))                         # sorted input no longer blows the recursion limit
a = big[:]; quick_sort_lomuto(a); assert a == big
a = big[:]; quick_sort_hoare(a); assert a == big

s = Stack(2)
assert s.is_empty() and s.pop() is None and s.peek() is None
assert s.push(1) and s.push(2) and not s.push(3) and s.is_full()
assert s.peek() == 2 and s.pop() == 2 and s.pop() == 1 and s.is_empty()

q = CQueue(3)
assert q.dequeue() is None and q.display() == []
assert q.enqueue(1) and q.enqueue(2) and q.enqueue(3) and not q.enqueue(4)
assert q.display() == [1, 2, 3]
assert q.dequeue() == 1 and q.enqueue(4) and q.display() == [2, 3, 4] and q.is_full()

def ll_items(ll):
    out, c = [], ll.root
    while c is not None:
        out.append(c.data); c = c.next
    return out

ll = LinkedList()
for v in (5, 1, 3, 3, 9):
    ll.insert_sorted(LinkedListNode(v))
assert ll_items(ll) == [1, 3, 3, 5, 9]
assert ll.delete(9) and ll.delete(1) and ll.delete(3) and not ll.delete(42)
assert ll_items(ll) == [3, 5]

def abll_items(ab):
    out, c = [], ab.root
    while c != -1:
        out.append(ab.array[c].data); c = ab.array[c].next
    return out

ab = ArrayBasedLinkedList(4)
assert all(ab.insert_sorted(v) for v in (5, 1, 3, 3))
assert ab.is_full() and not ab.insert_sorted(7)
assert abll_items(ab) == [1, 3, 3, 5]
assert ab.delete(1) and ab.delete(5) and not ab.delete(99)
assert abll_items(ab) == [3, 3] and not ab.is_full()
assert ab.insert_sorted(2) and abll_items(ab) == [2, 3, 3]
assert ab.delete(2) and ab.delete(3) and ab.delete(3) and ab.root == -1

for Tree in (BinarySearchTreeIter, BinarySearchTreeRec):
    t = Tree()
    assert t.inorder() == [] and t.search(1) is None
    for v in (8, 3, 10, 1, 6, 14, 4, 7, 13):
        t.insert(TreeNode(v))
    assert t.inorder() == [1, 3, 4, 6, 7, 8, 10, 13, 14]
    assert t.preorder() == [8, 3, 1, 6, 4, 7, 10, 14, 13]
    assert t.postorder() == [1, 4, 7, 6, 3, 13, 14, 10, 8]
    assert t.search(6).data == 6 and t.search(99) is None

for H in (HashTableLP, HashTableSC):
    h = H(3)
    assert h.insert("a", 0) and h.search("a") == 0 and h.search("zzz") is None
    h.insert("a", 5); assert h.search("a") == 5
h = HashTableLP(2)
assert h.insert("x", 1) and h.insert("y", 2) and not h.insert("z", 3)

print("All sanity tests passed")